# yolov5n-coco sur voc — inférence

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lrivals/EmbeddedComputervision/blob/main/notebooks/voc/yolov5n-coco_infer.ipynb)

Rôle : inférence ; palier **R** au premier passage (`SUBSET = 50`), **N** sur le split complet (4952 images, `SUBSET = 0`). Tâches : [T11.2](../../docs/tasks/M11-jeux-de-donnees.md), [T33.7](../../docs/tasks/M33-yolov5n.md) ;
notebooks : [M14](../../docs/tasks/M14-notebooks.md). Poids COCO hors domaine : seules les classes communes sont évaluées.

Prérequis :
- données : `data/VOCdevkit/VOC2007/ImageSets/Main/test.txt` — tools/get_voc.sh
- poids : `weights/ultralytics/yolov5n.pt` — Ultralytics checkpoint, AGPL-3.0, local only (neither on Drive nor fetched by tools/get_weights.sh): copy it to weights/ultralytics/ (Colab: upload it)

Chaque étape affiche la commande `python tools/…` qu'elle lance : un résultat se
reproduit en ligne de commande. Sorties dans `build/notebooks/voc/yolov5n-coco/`, jamais dans `results/`
(une mAP sur `SUBSET` images n'est pas publiable, règles de M12). Notebook généré par
`python -m tools.notebooks` (`make notebooks`) : modifier le gabarit
`tools/notebooks/gabarits.py`, pas ce fichier.

## Paramètres

In [ ]:
NET       = 'yolov5n-coco'  # nom de CFG_FILES ou cfg (relative à la racine)
DATASET   = 'voc'  # clé de DATASETS
SPLIT     = None  # None : split d'évaluation du jeu
WEIGHTS   = 'weights/ultralytics/yolov5n.pt'  # poids évalués
DEVICE    = 'cpu'  # cpu | gpu (entraînement seulement ; le reste sur CPU)
SUBSET    = 50  # images évaluées ; 0 : split complet (palier N)
SIZE      = None  # entrée LxH (ex. '640x192') ; None : celle de la cfg
RESIZE    = 'v5'  # v5 (letterbox and thresholds of YOLOv5, T33.4)
METRIC    = None  # None : celle du jeu (coco pour coco et flir)
DATA_ROOT = None  # racine du jeu ; None : data/<jeu> (Colab : dossier Drive)
DRIVE_DIR = '/content/drive/MyDrive/EmbeddedCV'  # Colab : archives data/<jeu>.tar ; None : pas de Drive
JOBS      = None  # processus des évaluations ; None : un par cœur CPU, plafonné par la mémoire (runtime TPU de Colab : des dizaines)
SHOW      = 6  # images affichées
OUT       = 'build/notebooks/voc/yolov5n-coco'  # sorties du notebook
REPO_URL  = 'https://github.com/lrivals/EmbeddedComputervision.git'  # Colab : dépôt cloné
REV       = 'main'  # Colab : révision

## Environnement

Local : rien n'est installé ni téléchargé, la cellule vérifie les prérequis et
s'arrête avec la commande à lancer. Colab : clone du dépôt, installation, CuPy si
`DEVICE = "gpu"`, poids, puis données par `colab.prepare_data` (archive
`<DRIVE_DIR>/data/<jeu>.tar` ou téléchargement direct ; ExDark et FLIR : archive
seulement, faite sur le PC par `tools/get_datasets.sh pack <jeu>`).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

COLAB = "google.colab" in sys.modules


def repo_root():
    for d in (Path.cwd(), *Path.cwd().parents):
        if (d / "tools" / "notebooks" / "commandes.py").exists():
            return d
    return None


ROOT = repo_root()
if COLAB:
    if ROOT is None:
        ROOT = Path("/content/EmbeddedComputervision")
        if not ROOT.exists():
            subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
    # Clone d'une session précédente : remis à REV (notebook et code du même commit).
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin", REV], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", "FETCH_HEAD"], check=True)
    for m in [m for m in sys.modules if m.split(".")[0] in ("tools", "yolo")]:
        del sys.modules[m]  # modules importés avant la mise à jour
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                    f"{ROOT}/python[data,plots]"], check=True)
    if DEVICE == "gpu":
        # Runtime CPU : pas de pilote, CuPy dirait « cudaErrorInsufficientDriver ».
        try:
            smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        except FileNotFoundError:
            smi = None
        if smi is None or smi.returncode != 0:
            raise RuntimeError("runtime Colab sans GPU : créer un nouveau serveur Colab "
                               "de type GPU (T4, L4 ou A100), ou DEVICE = 'cpu'")
        cuda = smi.stdout.split("CUDA Version:")[-1].split()[0]  # version du pilote
        cupy_pkg = "cupy-cuda13x" if int(cuda.split(".")[0]) >= 13 else "cupy-cuda12x"
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", cupy_pkg],
                       check=True)
elif ROOT is None:
    raise RuntimeError("racine du dépôt introuvable : ouvrir le notebook depuis notebooks/")
os.chdir(ROOT)
for p in (ROOT, ROOT / "python"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from tools.notebooks import commandes as C
from tools.notebooks import env

TRACE = env.trace(DEVICE)
env.check_device(DEVICE)
env.prepare(DATASET, WEIGHTS, NET, DATA_ROOT, COLAB, hint='Ultralytics checkpoint, AGPL-3.0, local only (neither on Drive nor fetched by tools/get_weights.sh): copy it to weights/ultralytics/ (Colab: upload it)',
            drive_dir=DRIVE_DIR)
Path(OUT).mkdir(parents=True, exist_ok=True)

## Réseau et poids

`yolo.models.tiny_yolo.build` et `yolo.io.darknet_weights.load_darknet_weights`.

In [ ]:
import json

import numpy as np
from IPython.display import Markdown, display
from PIL import Image

from tools.detect import draw
from tools.notebooks.matrice import SPLIT_IMAGES, palier
from yolo.data import datasets
from yolo.data.letterbox import input_size, parse_size, size_label
from yolo.infer.pipeline import detect, preprocess
from tools.eval_voc import load_net

net = load_net(NET, WEIGHTS)
size = input_size(net.net, parse_size(SIZE) if SIZE else None)
channels = net.net["input"][0]
family, names = datasets.model_family(DATASET, net.net["classes"])
print(f"{NET} : {net.net['classes']} classes ({family}), entrée {size_label(size)}, "
      f"{channels} canal(aux) ; poids {WEIGHTS}")

## Détections sur quelques images

`yolo.infer.pipeline.preprocess` et `detect` (seuil de la démo), boîtes de
`tools/detect.py:draw`.

In [ ]:
for s in datasets.load(DATASET, DATA_ROOT, SPLIT)[:SHOW]:
    with Image.open(s["image"]) as img:
        x, wh = preprocess(img, size, RESIZE, "pil", channels)
        boxes, scores, labels = detect(net, x[None], [wh], RESIZE)[0]
        shown = draw(img, boxes, scores, labels, names)
    shown.thumbnail((640, 640))
    print(f"{s['id']} : {len(labels)} détections")
    display(shown)

## Correspondance des classes (hors domaine, T11.2)

`MAPPINGS` de `yolo.data.datasets`, telle quelle ; les classes du jeu sans
équivalent sont retirées des vérités, les détections des autres classes du
modèle sont ignorées (`eval_view`).

In [ ]:
view = datasets.eval_view(DATASET, net.net["classes"])
mapping = datasets.MAPPINGS[(DATASET, family)]
rows = ["| classe du jeu | classe du modèle |", "|---|---|"]
rows += [f"| {c} | {mapping.get(c, '— (sans équivalent, ignorée)')} |"
         for c in datasets.DATASETS[DATASET].classes]
display(Markdown("\n".join(rows)))
print(f"classes évaluées ({len(view.names)}) : {', '.join(view.names)}")
print(f"classes du modèle ignorées : {len(names) - len(view.names)}")

## mAP flottante (T14.3)

`tools/eval_voc.py`, métrique du jeu (`Dataset.metric` ; COCO et FLIR : AP@[.5:.95]
et AP50). Courbes PR du mode automatique de M13 (métrique VOC).

In [ ]:
print(f"{SUBSET or 'toutes les'} images : palier "
      f"{palier(SUBSET or SPLIT_IMAGES.get(DATASET))} (règles de M12)")
map_float = Path(OUT) / "map_float.md"
map_float.unlink(missing_ok=True)
C.run(C.cmd_eval_voc(NET, WEIGHTS, DATASET, RESIZE, SUBSET, METRIC, SPLIT, SIZE,
                     DATA_ROOT, out=f"{OUT}/dets", markdown=map_float, jobs=JOBS))
display(Markdown(map_float.read_text()))
for png in sorted(Path(OUT, "dets", "figures").glob("pr*.png")):
    display(Image.open(png))

## Résumé

In [ ]:
print(map_float.read_text())
print(f"sorties : {OUT}/ ; révision {TRACE['rev']}")
print("commandes équivalentes :")
for c in C.HISTORY:
    print("  " + c)